In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
import seaborn as sns
import matplotlib.pyplot as plt




In [13]:
#1. Load Data

df = pd.read_csv('/content/insurance.csv') # Replace with your actual CSV file path
print("Original Data Shape:", df.shape)
print(df.head())


Original Data Shape: (1338, 7)
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [14]:
#2. Basic Info

print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())



Data Types:
 age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

Missing Values:
 age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


In [15]:
#3. Handle Missing Values

#Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

#Numerical: Replace with mean
num_imputer = SimpleImputer(strategy='mean')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

#Categorical: Replace with mode
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [18]:
#4. Encode Categorical Variables

#Label Encoding (for binary categorical)
le = LabelEncoder()
for col in cat_cols:
 if df[col].nunique() == 2:
  df[col] = le.fit_transform(df[col])

# One-hot Encoding (for more than 2 categories)
df = pd.get_dummies(df, columns=[col for col in cat_cols if df[col].nunique() > 2], drop_first=True)



In [19]:
#5. Outlier Detection and Treatment (using IQR)

for col in num_cols:
 Q1 = df[col].quantile(0.25)
 Q3 = df[col].quantile(0.75)
 IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df[col] = np.where(df[col] < lower, lower, df[col])
df[col] = np.where(df[col] > upper, upper, df[col])


In [25]:
# 6.Feature Scaling

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])



In [28]:
from sklearn.feature_selection import SelectKBest, f_regression
target_col = 'charges'  # Replace with your actual target column name

if target_col in df.columns:
    X = df.drop(target_col, axis=1)
    y = df[target_col]

    bestfeatures = SelectKBest(score_func=f_regression, k=10)
    fit = bestfeatures.fit(X, y)

    selected_features = X.columns[fit.get_support()]
    df = df[selected_features.to_list() + [target_col]]

    print("\nSelected Features:", selected_features.to_list())

corr = df.corr()
corr.sort_values(by='charges', ascending=False)


Selected Features: ['age', 'sex', 'bmi', 'children', 'smoker', 'region_northwest', 'region_southeast', 'region_southwest']


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=10 is greater than n_features=8. All the features will be returned.
  warnings.warn(


,age,sex,bmi,children,smoker,region_northwest,region_southeast,region_southwest,charges
charges,0.313394,0.052322,0.160175,0.073074,0.785958,-0.035204,0.059990,-0.044267,1.000000
smoker,-0.025019,0.076185,0.003750,0.007673,1.000000,-0.036945,0.068498,-0.036945,0.785958
age,1.000000,-0.020856,0.109272,0.042469,-0.025019,-0.000407,-0.011642,0.010016,0.313394
bmi,0.109272,0.046371,1.000000,0.012759,0.003750,-0.135996,0.270025,-0.006205,0.160175
children,0.042469,0.017163,0.012759,1.000000,0.007673,0.024806,-0.023066,0.021914,0.073074
region_southeast,-0.011642,0.017117,0.270025,-0.023066,0.068498,-0.346265,1.000000,-0.346265,0.059990
sex,-0.020856,1.000000,0.046371,0.017163,0.076185,-0.011156,0.017117,-0.004184,0.052322
region_northwest,-0.000407,-0.011156,-0.135996,0.024806,-0.036945,1.000000,-0.346265,-0.320829,-0.035204
region_southwest,0.010016,-0.004184,-0.006205,0.021914,-0.036945,-0.320829,-0.346265,1.000000,-0.044267


In [30]:
import numpy as np
import sklearn.preprocessing as preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df.drop(columns=['charges'])
y = np.asarray(df['charges'])

# Scale the features using preprocessing.StandardScaler()
X = preprocessing.StandardScaler().fit(X).transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size = 0.3, random_state = 100)

print ('Train set:', X_train.shape, y_train.shape)
print ('Test set:', X_test.shape, y_test.shape)

Train set: (936, 8) (936,)
Test set: (402, 8) (402,)


In [31]:
#1. Linear Regression

from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))


R² Score: 0.7895063962914661


In [32]:
#2. Decision Tree

from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))


R² Score: 0.6505436353857904


In [33]:
#3. Random Forest
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))


R² Score: 0.8220143625690588


In [34]:
#4. K-Nearest Neighbors

from sklearn.neighbors import KNeighborsRegressor
model = KNeighborsRegressor(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))


R² Score: 0.7966680752601463


In [35]:
#5. Support Vector Regression

from sklearn.svm import SVR
model = SVR()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))


R² Score: 0.8514335322111817


In [36]:
#6. Gradient Boosting

from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))


R² Score: 0.8591886146570358


In [37]:
#7. Ridge Regression

from sklearn.linear_model import Ridge
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import r2_score
print("R² Score:", r2_score(y_test, y_pred))

R² Score: 0.789516922500116
